# Session 3 — Cleaning Data

Runnable code for the **Build It** and **Experiment** sections.
Run each cell top to bottom.

In [1]:
import io

import numpy as np
import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 3.0.3


## 1. Build It — Core Code

### 1.1 A deliberately messy frame

In [2]:
messy = pd.read_csv(
    io.StringIO(
        """name,city,price,units,joined,email
  ana ,berlin,12.5,3,2023-01-05,ANA@Example.com
BO,Paris,7.5,,2023-02-10,bo@example.com
Cara,lisbon,n/a,5,not-a-date,
dan,BERLIN,22.0,2,2023-03-15,dan@example.com
ana,berlin,12.5,3,2023-01-05,ANA@Example.com
Eve,,15.0,4,2023-04-01,eve@example.com
"""
    )
)
messy

,name,city,price,units,joined,email
0,ana,berlin,12.5,3.0,2023-01-05,ANA@Example.com
1,BO,Paris,7.5,NaN,2023-02-10,bo@example.com
2,Cara,lisbon,NaN,5.0,not-a-date,NaN
3,dan,BERLIN,22.0,2.0,2023-03-15,dan@example.com
4,ana,berlin,12.5,3.0,2023-01-05,ANA@Example.com
5,Eve,NaN,15.0,4.0,2023-04-01,eve@example.com


### 1.2 Finding missing values

In [3]:
print("isna() is a boolean frame:")
print(messy.isna().head(3))
print("\nmissing per column:")
print(messy.isna().sum())

isna() is a boolean frame:
    name   city  price  units  joined  email
0  False  False  False  False   False  False
1  False  False  False   True   False  False
2  False  False   True  False   False   True

missing per column:
name      0
city      1
price     1
units     1
joined    0
email     1
dtype: int64


In [4]:
print("fully populated columns:", messy.notna().all().to_dict())

fully populated columns: {'name': True, 'city': False, 'price': False, 'units': False, 'joined': True, 'email': False}


### 1.3 Dropping missing values

In [5]:
no_missing_price = messy.dropna(subset=["price"])
print("rows after dropna(subset=['price']):", len(no_missing_price))

no_empty_rows = messy.dropna(how="all")
print("rows after dropna(how='all')       :", len(no_empty_rows))

rows after dropna(subset=['price']): 5
rows after dropna(how='all')       : 6


### 1.4 Filling missing values (and reassigning — CoW-safe)

In [6]:
clean = messy.copy()
clean["units"] = clean["units"].fillna(0)
clean["price"] = pd.to_numeric(clean["price"], errors="coerce")
clean["price"] = clean["price"].fillna(clean["price"].mean())
clean[["name", "price", "units"]]

,name,price,units
0,ana,12.5,3.0
1,BO,7.5,0.0
2,Cara,13.9,5.0
3,dan,22.0,2.0
4,ana,12.5,3.0
5,Eve,15.0,4.0


### 1.5 Converting dtypes

In [7]:
print("before:", messy["price"].dtype, "|", messy["joined"].dtype)

typed = messy.copy()
typed["price"] = pd.to_numeric(typed["price"], errors="coerce")
typed["joined"] = pd.to_datetime(typed["joined"], errors="coerce")
typed["city"] = typed["city"].astype("category")
print("after :", typed["price"].dtype, "|", typed["joined"].dtype, "|", typed["city"].dtype)

before: float64 | str
after : float64 | datetime64[us] | category


In [8]:
# category is compact: codes plus a lookup table
print("categories:", list(typed["city"].cat.categories))
typed["city"].cat.codes

categories: ['BERLIN', 'Paris', 'berlin', 'lisbon']


0    2
1    1
2    3
3    0
4    2
5   -1
dtype: int8

### 1.6 Duplicates

In [9]:
print("duplicated rows:", messy.duplicated().sum())

deduped = messy.drop_duplicates()
print("after drop_duplicates:", len(deduped), "of", len(messy))

duplicated rows: 0
after drop_duplicates: 6 of 6


In [10]:
# keep the last occurrence instead of the first
keep_last = messy.drop_duplicates(subset=["name", "joined"], keep="last")
len(keep_last)

6

### 1.7 String methods with `.str`

In [11]:
text = messy.copy()
text["name"] = text["name"].str.strip().str.title()
text["city"] = text["city"].str.strip().str.title()
text["email"] = text["email"].str.lower()
text[["name", "city", "email"]]

,name,city,email
0,Ana,Berlin,ana@example.com
1,Bo,Paris,bo@example.com
2,Cara,Lisbon,NaN
3,Dan,Berlin,dan@example.com
4,Ana,Berlin,ana@example.com
5,Eve,NaN,eve@example.com


In [12]:
print("contains '@' rows:", int(messy["email"].str.contains("@", na=False).sum()))
print("name lengths     :", messy["name"].str.strip().str.len().tolist())
print("split names      :")
print(messy["name"].str.strip().str.split(" ", n=1, expand=True))

contains '@' rows: 5
name lengths     : [3, 2, 4, 3, 3, 3]
split names      :
      0
0   ana
1    BO
2  Cara
3   dan
4   ana
5   Eve


### 1.8 Rename and replace

In [13]:
renamed = messy.rename(columns={"name": "student", "city": "town"})
renamed.columns.tolist()

['student', 'town', 'price', 'units', 'joined', 'email']

In [14]:
replaced = messy.copy()
replaced["city"] = replaced["city"].str.strip().replace({"": np.nan})
replaced["city"] = replaced["city"].fillna("Unknown")
replaced["city"].tolist()

['berlin', 'Paris', 'lisbon', 'BERLIN', 'berlin', 'Unknown']

## 2. Experiment

Change a parameter, predict the output, then run the cell.

**Experiment 1** — a missing integer turns the whole column into floats.

In [15]:
ints = pd.DataFrame({"n": [1, 2, None]})
print("dtype with missing int:", ints["n"].dtype)

dtype with missing int: float64


**Experiment 2** — successful fills let pandas keep an integer dtype.

In [16]:
ok = pd.DataFrame({"n": [1, 2, None]})
ok["n"] = ok["n"].fillna(0).astype("int64")
print("after fill + astype:", ok["n"].dtype, ok["n"].tolist())

after fill + astype: int64 [1, 2, 0]


**Experiment 3** — `astype` raises on bad text, `to_numeric(..., errors='coerce')` does not.

In [17]:
try:
    pd.Series(["1", "n/a", "3"]).astype("float64")
except ValueError as err:
    print("astype -> ValueError:", err)

print("to_numeric coerce ->", pd.to_numeric(pd.Series(["1", "n/a", "3"]), errors="coerce").tolist())

astype -> ValueError: could not convert string to float: 'n/a'
to_numeric coerce -> [1.0, nan, 3.0]


**Experiment 4** — forward-fill versus backward-fill.

In [18]:
holed = pd.Series([1.0, np.nan, np.nan, 4.0])
print("ffill:", holed.ffill().tolist())
print("bfill:", holed.bfill().tolist())

ffill: [1.0, 1.0, 1.0, 4.0]
bfill: [1.0, 4.0, 4.0, 4.0]


**Experiment 5** — pass `na=False` to `.str.contains` for predictable masks.
On an object-dtype column the default leaves `NaN` for missing entries, which cannot
be used as a boolean mask; `na=False` turns those entries into `False`.

In [19]:
emails = pd.Series(["a@x.com", None, "b@y.com"], dtype="object")
print("default (NaN):", emails.str.contains("@").tolist())
print("with na=False:", emails.str.contains("@", na=False).tolist())

default (NaN): [True, None, True]
with na=False: [True, False, True]


## 3. Mini Project — Messy CSV Cleanup

In [20]:
raw = pd.read_csv(
    io.StringIO(
        """name,city,signup,spend,visits
  Ana ,berlin,2023-01-05,12.50,3
BO,Paris,2023-02-10,7.5,
Cara,lisbon,not-a-date,n/a,5
dan,BERLIN,2023-03-15,22.00,2
Ana,berlin,2023-01-05,12.50,3
Eve,,2023-04-01,15.0,4
"""
    )
)
print("raw shape:", raw.shape)
raw

raw shape: (6, 5)


,name,city,signup,spend,visits
0,Ana,berlin,2023-01-05,12.5,3.0
1,BO,Paris,2023-02-10,7.5,NaN
2,Cara,lisbon,not-a-date,NaN,5.0
3,dan,BERLIN,2023-03-15,22.0,2.0
4,Ana,berlin,2023-01-05,12.5,3.0
5,Eve,NaN,2023-04-01,15.0,4.0


In [21]:
# Step 2 — tidy the column labels.
df = raw.rename(columns={"signup": "joined", "spend": "amount"})

# Step 3 — strip / title-case text, then mark and fill empty cities.
df["name"] = df["name"].str.strip().str.title()
df["city"] = df["city"].str.strip().str.title().replace({"": np.nan})
df["city"] = df["city"].fillna("Unknown")

# Step 4 — convert the messy typed columns.
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df

,name,city,joined,amount,visits
0,Ana,Berlin,2023-01-05,12.5,3.0
1,Bo,Paris,2023-02-10,7.5,NaN
2,Cara,Lisbon,NaT,NaN,5.0
3,Dan,Berlin,2023-03-15,22.0,2.0
4,Ana,Berlin,2023-01-05,12.5,3.0
5,Eve,Unknown,2023-04-01,15.0,4.0


In [22]:
# Step 5 — report then fill remaining missing values.
print("missing before fill:")
print(df.isna().sum())

df["visits"] = df["visits"].fillna(0).astype("int64")
df["amount"] = df["amount"].fillna(df["amount"].mean())

print("\nmissing after fill:")
print(df.isna().sum())
df

missing before fill:
name      0
city      0
joined    1
amount    1
visits    1
dtype: int64

missing after fill:
name      0
city      0
joined    1
amount    0
visits    0
dtype: int64


,name,city,joined,amount,visits
0,Ana,Berlin,2023-01-05,12.5,3
1,Bo,Paris,2023-02-10,7.5,0
2,Cara,Lisbon,NaT,13.9,5
3,Dan,Berlin,2023-03-15,22.0,2
4,Ana,Berlin,2023-01-05,12.5,3
5,Eve,Unknown,2023-04-01,15.0,4


In [23]:
# Step 6 — remove duplicates keyed on name + joined.
print("duplicates:", int(df.duplicated(subset=["name", "joined"]).sum()))
df = df.drop_duplicates(subset=["name", "joined"], keep="first")

# Step 7 — make city categorical and show the result.
df["city"] = df["city"].astype("category")
print("\ndtypes:")
print(df.dtypes)
print("\nclean shape:", df.shape)
df

duplicates: 1

dtypes:
name                 str
city            category
joined    datetime64[us]
amount           float64
visits             int64
dtype: object

clean shape: (5, 5)


,name,city,joined,amount,visits
0,Ana,Berlin,2023-01-05,12.5,3
1,Bo,Paris,2023-02-10,7.5,0
2,Cara,Lisbon,NaT,13.9,5
3,Dan,Berlin,2023-03-15,22.0,2
5,Eve,Unknown,2023-04-01,15.0,4
